In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การจำลองวงจร stabilizer อย่างมีประสิทธิภาพด้วย Qiskit Aer primitives

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
หน้านี้แสดงวิธีใช้ Qiskit Aer primitives เพื่อจำลอง stabilizer circuits อย่างมีประสิทธิภาพ รวมถึง Circuit ที่อยู่ภายใต้สัญญาณรบกวน Pauli

Stabilizer circuits หรือที่รู้จักในชื่อ Clifford circuits เป็นประเภทย่อยที่สำคัญของวงจรควอนตัมที่สามารถจำลองแบบคลาสสิกได้อย่างมีประสิทธิภาพ มีหลายวิธีที่เทียบเท่ากันในการนิยาม stabilizer circuits โดยนิยามหนึ่งคือ stabilizer circuit คือวงจรควอนตัมที่ประกอบด้วยเฉพาะ Gate ต่อไปนี้:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Measurement](../api/qiskit/circuit#qiskit.circuit.Measure)

โปรดทราบว่าด้วย Hadamard และ S เราสามารถสร้าง Pauli rotation Gate ใดๆ ([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate) และ [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)) ที่มีมุมอยู่ในเซต ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ (ถึงเฟสโกลบอล) ดังนั้นเราจึงรวม Gate เหล่านี้ในนิยามได้เช่นกัน

Stabilizer circuits มีความสำคัญต่อการศึกษาการแก้ไขข้อผิดพลาดควอนตัม การจำลองแบบคลาสสิกได้ยังทำให้มีประโยชน์สำหรับการตรวจสอบผลลัพธ์ของคอมพิวเตอร์ควอนตัม ตัวอย่างเช่น สมมติว่าคุณต้องการรัน Circuit ควอนตัมที่ใช้ 100 Qubit บนคอมพิวเตอร์ควอนตัม คุณจะรู้ได้อย่างไรว่าคอมพิวเตอร์ควอนตัมทำงานถูกต้อง? Circuit ควอนตัมบน 100 Qubit อยู่เกินขีดความสามารถของการจำลองแบบ brute-force คลาสสิก ด้วยการปรับ Circuit ของคุณให้กลายเป็น stabilizer circuit คุณสามารถรัน Circuit บนคอมพิวเตอร์ควอนตัมที่มีโครงสร้างคล้ายกับ Circuit ที่ต้องการ แต่สามารถจำลองบนคอมพิวเตอร์คลาสสิกได้ ด้วยการตรวจสอบผลลัพธ์ของคอมพิวเตอร์ควอนตัมบน stabilizer circuits คุณสามารถสร้างความมั่นใจว่ามันทำงานถูกต้องบน non-stabilizer circuits ด้วย ดู [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3) สำหรับตัวอย่างของแนวคิดนี้ในทางปฏิบัติ

[การจำลองแบบแม่นยำและมีสัญญาณรบกวนด้วย Qiskit Aer primitives](simulate-with-qiskit-aer) แสดงวิธีใช้ [Qiskit Aer](https://qiskit.org/ecosystem/aer/) เพื่อทำการจำลองแบบแม่นยำและมีสัญญาณรบกวนของ Circuit ควอนตัมทั่วไป พิจารณา Circuit ตัวอย่างที่ใช้ในบทความนั้น ซึ่งเป็น Circuit 8 Qubit ที่สร้างโดยใช้ [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2):

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

ด้วย Qiskit Aer เราสามารถจำลอง Circuit นี้ได้อย่างง่ายดาย แต่สมมติว่าเราตั้งจำนวน Qubit เป็น 500:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

เนื่องจากต้นทุนในการจำลอง Circuit ควอนตัมเพิ่มขึ้นแบบ exponential ตามจำนวน Qubit Circuit ขนาดใหญ่เช่นนี้โดยทั่วไปจะเกินความสามารถของ simulator แม้แต่ simulator ประสิทธิภาพสูงอย่าง Qiskit Aer การจำลองแบบคลาสสิกของ Circuit ควอนตัมทั่วไปจะไม่สามารถทำได้เมื่อจำนวน Qubit เกินประมาณ 50 ถึง 100 Qubit อย่างไรก็ตาม โปรดทราบว่า Circuit efficient_su2 มีพารามิเตอร์เป็นมุมบน $R_y$ และ $R_z$ Gates หากมุมทั้งหมดอยู่ในเซต ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ Circuit นั้นก็คือ stabilizer circuit และสามารถจำลองได้อย่างมีประสิทธิภาพ!

ใน cell ต่อไปนี้ เราจะรัน Circuit ด้วย Sampler primitive ที่รองรับโดย stabilizer circuit simulator โดยใช้พารามิเตอร์ที่เลือกแบบสุ่มเพื่อรับประกันว่า Circuit เป็น stabilizer circuit

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Stabilizer circuit simulator ยังรองรับการจำลองแบบมีสัญญาณรบกวน แต่เฉพาะสำหรับประเภทย่อยของ noise models เท่านั้น โดยเฉพาะ สัญญาณรบกวนควอนตัมต้องถูกกำหนดลักษณะโดย [Pauli error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error) channel [Depolarizing error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html) อยู่ในหมวดหมู่นี้จึงสามารถจำลองได้เช่นกัน noise channels แบบคลาสสิกอย่าง [readout error](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html) ก็สามารถจำลองได้เช่นกัน

cell โค้ดต่อไปนี้รันการจำลองเดียวกับก่อน แต่คราวนี้ระบุ noise model ที่เพิ่ม depolarizing error 2% ให้กับแต่ละ CX Gate รวมถึง readout error ที่พลิก bit ที่วัดแต่ละ bit ด้วยความน่าจะเป็น 5%

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

ตอนนี้ มาใช้ Estimator primitive ที่รองรับโดย stabilizer simulator เพื่อคำนวณค่าคาดหวังของ observable $ZZ \cdots Z$ เนื่องจากโครงสร้างพิเศษของ stabilizer circuits ผลลัพธ์มีแนวโน้มสูงที่จะเป็น 0

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>